# Surface–Columnar Linkage: Publication Figures

10 publication-quality figures for the surface-columnar aerosol analysis at Addis Ababa.

1. Sequential exclusion scatter (Eom et al.)
2. *w* parameter seasonal breakdown
3. BC vs Fine Soil scatter
4. AOD binning (Segura et al.)
5. SSA spectral shape, dSSA, w vs SSA
6. η decomposition (Snider et al.)
7. HIPS/FTIR ratio vs AOD
8. HIPS vs FTIR colored by iron
9. Control sites HIPS/FTIR vs AOD
10. Season-separated HIPS vs FTIR regressions

In [ ]:
import sys, os, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from scipy import stats
from pathlib import Path

FTIR_HIPS_DIR = Path('/Users/ahmadjalil/github/aethmodular/research/ftir_hips_chem')
scripts_path = str(FTIR_HIPS_DIR / 'scripts')
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

from config import SITES, MAC_VALUE, FILTER_DATA_PATH
from data_matching import load_filter_data, pivot_filter_by_id

# Publication style
plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10,
    'xtick.labelsize': 9, 'ytick.labelsize': 9, 'legend.fontsize': 8,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3, 'grid.linewidth': 0.5,
    'axes.spines.top': False, 'axes.spines.right': False,
})

SEASON_MAP = {10:'Dry',11:'Dry',12:'Dry',1:'Dry',2:'Dry',
              3:'Belg',4:'Belg',5:'Belg',6:'Kiremt',7:'Kiremt',8:'Kiremt',9:'Kiremt'}
SO = ['Dry','Belg','Kiremt']
SC = {'Dry':'#E67E22','Belg':'#27AE60','Kiremt':'#3498DB'}
MISS = -999.

OUT = 'output/plots/pub_figures'
os.makedirs(OUT, exist_ok=True)
os.makedirs('data', exist_ok=True)
print('Setup complete')

## Data Loading

In [ ]:
# ── Aethalometer BC ──
bc_raw = pd.read_pickle(FTIR_HIPS_DIR / 'processed_sites/df_Addis_Ababa_9am_resampled.pkl')
bc_raw['datetime_local'] = pd.to_datetime(bc_raw['datetime_local'])
bc_raw.set_index('datetime_local', inplace=True)
bc_raw.sort_index(inplace=True)
for c in [col for col in bc_raw.columns if 'BCc' in col and 'smoothed' not in col]:
    bc_raw[c] = bc_raw[c] / 1000  # ng→µg
bc_raw.loc[bc_raw['IR BCc'] < 0, 'IR BCc'] = np.nan
mu, sig = bc_raw['IR BCc'].mean(), bc_raw['IR BCc'].std()
bc_raw.loc[bc_raw['IR BCc'] > mu + 3*sig, 'IR BCc'] = np.nan
bc_raw['Season'] = bc_raw.index.month.map(SEASON_MAP)

bc = bc_raw[['IR BCc','UV BCc','Season']].copy()
bc.index = bc.index.normalize()
if bc.index.tz is not None:
    bc.index = bc.index.tz_localize(None)
bc.index.name = 'Date'
print(f'BC: {bc["IR BCc"].notna().sum()} days')

In [ ]:
# ── AERONET ──
ADIR = ('/Users/ahmadjalil/Library/CloudStorage/GoogleDrive-ahzs645@gmail.com/'
        'My Drive/University/Research/Grad/UC Davis Ann/NASA MAIA/Data/AERONET/Jacros/'
        '20220101_20251231_AAU_Jackros_ET Daily/')

def load_aeronet(path, date_col='Date(dd:mm:yyyy)', skiprows=6):
    df = pd.read_csv(path, skiprows=skiprows)
    # Handle both column name variants
    if date_col not in df.columns:
        date_col = [c for c in df.columns if 'Date' in c and 'dd' in c][0]
    df['Date'] = pd.to_datetime(df[date_col], format='%d:%m:%Y')
    df.set_index('Date', inplace=True)
    df.sort_index(inplace=True)
    df.replace(MISS, np.nan, inplace=True)
    df['Season'] = df.index.month.map(SEASON_MAP)
    return df

aod = load_aeronet(ADIR + '20220101_20251231_AAU_Jackros_ET.lev20')
sda = load_aeronet(ADIR + '20220101_20251231_AAU_Jackros_ET.ONEILL_lev20',
                   date_col='Date_(dd:mm:yyyy)')
ssa = load_aeronet(ADIR + '20220101_20251231_AAU_Jackros_ET.SSA_ALM_lev15_daily')

ssa_rename = {'Single_Scattering_Albedo[440nm]':'SSA_440',
              'Single_Scattering_Albedo[675nm]':'SSA_675',
              'Single_Scattering_Albedo[870nm]':'SSA_870',
              'Single_Scattering_Albedo[1020nm]':'SSA_1020'}
ssa.rename(columns=ssa_rename, inplace=True)
ssa['dSSA'] = ssa['SSA_440'] - ssa['SSA_870']

print(f'AOD: {len(aod)}d | SDA: {len(sda)}d | SSA: {len(ssa)}d')

In [ ]:
# ── Filter chemistry ──
filter_data = load_filter_data()

chem_params = [
    'EC_ftir','OC_ftir','HIPS_Fabs',
    'ChemSpec_Sulfate_Ion_PM2.5','ChemSpec_Nitrate_Ion_PM2.5',
    'ChemSpec_Ammonium_Ion_PM2.5','ChemSpec_Iron_PM2.5',
    'ChemSpec_Silicon_PM2.5','ChemSpec_Aluminum_PM2.5',
    'ChemSpec_Calcium_PM2.5','ChemSpec_Titanium_PM2.5',
    'ChemSpec_BC_PM2.5',
]
chem = pivot_filter_by_id(filter_data, 'ETAD', params=chem_params)
chem['date'] = pd.to_datetime(chem['date'])
rename = {'chemspec_sulfate_ion_pm2.5':'sulfate','chemspec_nitrate_ion_pm2.5':'nitrate',
          'chemspec_ammonium_ion_pm2.5':'ammonium','chemspec_iron_pm2.5':'iron',
          'chemspec_silicon_pm2.5':'silicon','chemspec_aluminum_pm2.5':'aluminum',
          'chemspec_calcium_pm2.5':'calcium','chemspec_titanium_pm2.5':'titanium',
          'chemspec_bc_pm2.5':'chemspec_bc'}
chem.rename(columns=rename, inplace=True)
chem['AS'] = chem['sulfate'] * 1.375
chem['AN'] = chem['nitrate'] * 1.290

# Fine soil (Malm et al. 1994): elemental data is in ng/m3, convert to ug/m3
# Don't fillna(0) — rows without ChemSpec data should stay NaN
chem['fine_soil'] = (2.20*chem['aluminum'] + 2.49*chem['silicon'] +
                     1.63*chem['calcium'] + 2.42*chem['iron'] +
                     1.94*chem['titanium']) / 1000  # ng/m3 -> ug/m3

chem['hips_bc'] = chem['hips_fabs'] / MAC_VALUE if 'hips_fabs' in chem.columns else np.nan
print(f'Filter chemistry: {len(chem)} filters')
print(f'  Filters with fine_soil: {chem["fine_soil"].notna().sum()}')
print(f'  fine_soil range: {chem["fine_soil"].min():.2f} – {chem["fine_soil"].max():.2f} ug/m3')

In [ ]:
# ── Master merge ──
m = bc[['IR BCc','UV BCc','Season']].copy()
m = m.join(aod[['AOD_440nm','AOD_500nm','AOD_675nm','AOD_870nm','440-870_Angstrom_Exponent']], how='left')
m = m.join(sda[['FineModeFraction_500nm[eta]','Fine_Mode_AOD_500nm[tau_f]','Coarse_Mode_AOD_500nm[tau_c]']], how='left')
m.rename(columns={'FineModeFraction_500nm[eta]':'FMF','Fine_Mode_AOD_500nm[tau_f]':'Fine_AOD',
                  'Coarse_Mode_AOD_500nm[tau_c]':'Coarse_AOD','440-870_Angstrom_Exponent':'AE'}, inplace=True)
m = m.join(ssa[['SSA_440','SSA_675','SSA_870','SSA_1020','dSSA']], how='left')

ci = chem.set_index('date')
ci.index = pd.to_datetime(ci.index)
ccols = [c for c in ['ftir_ec','hips_fabs','hips_bc','sulfate','nitrate','ammonium',
                      'iron','silicon','aluminum','fine_soil','AS','AN','chemspec_bc']
         if c in ci.columns]
m = m.join(ci[ccols], how='left')
m['BC_AOD'] = m['IR BCc'] / m['AOD_500nm']

# Precipitable water
pw_col = 'Precipitable_Water(cm)'
if pw_col in aod.columns:
    m = m.join(aod[[pw_col]], how='left')

print(f'Master: {len(m)} days, BC+AOD={m[["IR BCc","AOD_500nm"]].dropna().shape[0]}, '
      f'SSA={m["SSA_440"].notna().sum()}, chem={m[ccols].dropna(how="all").shape[0]}')

---
## Figure 1: Sequential Exclusion Scatter (Eom et al.)

In [ ]:
df1 = m.dropna(subset=['IR BCc','AOD_500nm']).copy()
df1 = df1[df1['AOD_500nm'] > 0]

exclusions = [0, 20, 40, 60, 80]
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes_flat = axes.flatten()

for idx, ex in enumerate(exclusions):
    ax = axes_flat[idx]
    if ex == 0:
        sub = df1.copy()
    else:
        thresh = df1['BC_AOD'].quantile(ex / 100)
        sub = df1[df1['BC_AOD'] >= thresh].copy()
    
    for s in SO:
        d = sub[sub['Season'] == s]
        ax.scatter(d['IR BCc'], d['AOD_500nm'], s=18, alpha=0.45,
                  color=SC[s], edgecolors='k', linewidth=0.2, label=s, zorder=3)
    
    valid = sub.dropna(subset=['IR BCc','AOD_500nm'])
    if len(valid) >= 5:
        r, p = stats.pearsonr(valid['IR BCc'], valid['AOD_500nm'])
        z = np.polyfit(valid['IR BCc'], valid['AOD_500nm'], 1)
        xr = np.linspace(valid['IR BCc'].min(), valid['IR BCc'].max(), 50)
        ax.plot(xr, np.poly1d(z)(xr), 'k--', linewidth=1.5, alpha=0.7, zorder=4)
        ax.text(0.05, 0.95, f'r = {r:.3f}\nn = {len(valid)}',
               transform=ax.transAxes, va='top', fontsize=9, fontweight='bold',
               bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray', alpha=0.9))
    
    ax.set_title(f'Ex{ex}' + ('' if ex == 0 else f' (bottom {ex}% removed)'), fontweight='bold')
    ax.set_xlabel('BC ($\\mu$g/m$^3$)')
    ax.set_ylabel('AOD$_{500}$')
    if idx == 0:
        ax.legend(fontsize=8, loc='lower right')

# Panel 6: BC vs Fine AOD
ax = axes_flat[5]
valid_fa = df1.dropna(subset=['IR BCc','Fine_AOD'])
for s in SO:
    d = valid_fa[valid_fa['Season'] == s]
    ax.scatter(d['IR BCc'], d['Fine_AOD'], s=18, alpha=0.45,
              color=SC[s], edgecolors='k', linewidth=0.2, label=s, zorder=3)
if len(valid_fa) >= 5:
    r, p = stats.pearsonr(valid_fa['IR BCc'], valid_fa['Fine_AOD'])
    z = np.polyfit(valid_fa['IR BCc'], valid_fa['Fine_AOD'], 1)
    xr = np.linspace(valid_fa['IR BCc'].min(), valid_fa['IR BCc'].max(), 50)
    ax.plot(xr, np.poly1d(z)(xr), 'k--', linewidth=1.5, alpha=0.7, zorder=4)
    ax.text(0.05, 0.95, f'r = {r:.3f}\nn = {len(valid_fa)}',
           transform=ax.transAxes, va='top', fontsize=9, fontweight='bold',
           bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray', alpha=0.9))
ax.set_title('BC vs Fine AOD$_{500}$', fontweight='bold')
ax.set_xlabel('BC ($\\mu$g/m$^3$)')
ax.set_ylabel('Fine Mode AOD$_{500}$')
ax.legend(fontsize=8, loc='lower right')

plt.suptitle('Figure 1: Sequential Exclusion of Low BC/AOD Days (Eom et al. 2025)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUT}/fig01_sequential_exclusion.png')
plt.show()

---
## Figure 2: *w* Parameter Seasonal Breakdown

In [ ]:
# Compute w
dw = chem.dropna(subset=['AS','AN']).copy()
bc_w = 'chemspec_bc' if 'chemspec_bc' in dw.columns and dw['chemspec_bc'].notna().sum() > 5 else 'ftir_ec'
dw = dw.dropna(subset=[bc_w])
denom = dw['AS'] + dw['AN'] + dw[bc_w]
dw['w'] = (dw['AS'] + dw['AN']) / denom
dw = dw[denom > 0]
dw['Season'] = dw['date'].dt.month.map(SEASON_MAP)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# (a) w by season bar chart
ax = axes[0]
means, errs, ns = [], [], []
for s in SO:
    sub = dw[dw['Season'] == s]['w']
    means.append(sub.mean())
    errs.append(sub.std() / np.sqrt(len(sub)))
    ns.append(len(sub))
bars = ax.bar(SO, means, yerr=errs, capsize=5, color=[SC[s] for s in SO],
              edgecolor='black', linewidth=0.5, alpha=0.8, width=0.6)
for i, (n, bar) in enumerate(zip(ns, bars)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + errs[i] + 0.01,
            f'n={n}', ha='center', fontsize=9)
ax.set_ylabel('w = (AS + AN) / (AS + AN + BC)', fontsize=10)
ax.set_title('(a) Non-absorbing fraction by season', fontweight='bold')
ax.set_ylim(0, 0.6)

# (b) Stacked bar: BC, AS, AN
ax = axes[1]
sm = []
for s in SO:
    sub = dw[dw['Season'] == s]
    sm.append({'Season': s, 'BC': sub[bc_w].mean(), 'AS': sub['AS'].mean(), 'AN': sub['AN'].mean()})
sm = pd.DataFrame(sm)
x = np.arange(len(SO))
w = 0.55
ax.bar(x, sm['BC'], w, label='BC', color='#2C3E50', alpha=0.85)
ax.bar(x, sm['AS'], w, bottom=sm['BC'], label='Amm. Sulfate', color='#3498DB', alpha=0.85)
ax.bar(x, sm['AN'], w, bottom=sm['BC']+sm['AS'], label='Amm. Nitrate', color='#2ECC71', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(SO)
ax.set_ylabel('Concentration ($\\mu$g/m$^3$)', fontsize=10)
ax.set_title('(b) BC + AS + AN composition by season', fontweight='bold')
ax.legend(fontsize=9)

plt.suptitle('Figure 2: w Parameter — Non-absorbing Mass Fraction (Eom et al.)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT}/fig02_w_parameter.png')
plt.show()

---
## Figure 3: BC vs Fine Soil

In [ ]:
bc_col = 'ftir_ec' if 'ftir_ec' in chem.columns else 'hips_bc'
d3 = chem.dropna(subset=[bc_col, 'fine_soil']).copy()
d3['Season'] = d3['date'].dt.month.map(SEASON_MAP)

fig, ax = plt.subplots(figsize=(7, 6.5))
for s in SO:
    sub = d3[d3['Season'] == s]
    ax.scatter(sub['fine_soil'], sub[bc_col], s=40, alpha=0.6,
              color=SC[s], edgecolors='k', linewidth=0.3, label=s, zorder=3)

lim = max(d3['fine_soil'].max(), d3[bc_col].max()) * 1.1
ax.plot([0, lim], [0, lim], 'k--', linewidth=1.2, alpha=0.5, label='1:1 line', zorder=2)

if len(d3) >= 5:
    r, p = stats.pearsonr(d3['fine_soil'], d3[bc_col])
    ax.text(0.95, 0.05, f'r = {r:.3f}\nn = {len(d3)}',
           transform=ax.transAxes, ha='right', fontsize=10, fontweight='bold',
           bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray', alpha=0.9))

ax.set_xlabel('Fine Soil ($\\mu$g/m$^3$)', fontsize=11)
ax.set_ylabel(f'{bc_col.replace("_"," ").upper()} ($\\mu$g/m$^3$)', fontsize=11)
ax.set_title('Figure 3: BC vs Fine Soil — Addis Ababa is BC-dominated',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim(0, None)
ax.set_ylim(0, None)
ax.set_aspect('equal')

plt.tight_layout()
plt.savefig(f'{OUT}/fig03_bc_vs_fine_soil.png')
plt.show()

---
## Figure 4: AOD Binning (Segura et al.)

In [ ]:
d4 = m.dropna(subset=['IR BCc','AOD_500nm']).copy()
d4 = d4[d4['AOD_500nm'] > 0]

bw = 0.05
bins = np.arange(0, d4['AOD_500nm'].quantile(0.98) + bw, bw)
d4['AOD_bin'] = pd.cut(d4['AOD_500nm'], bins=bins)
d4['AOD_mid'] = d4['AOD_bin'].apply(lambda x: x.mid if pd.notna(x) else np.nan)

binned = d4.groupby('AOD_mid', observed=True).agg(
    BC_mean=('IR BCc','mean'), BC_se=('IR BCc', lambda x: x.std()/np.sqrt(len(x)) if len(x)>1 else np.nan),
    n=('IR BCc','count')).dropna()
bv = binned[binned['n'] >= 3]

r_daily = stats.pearsonr(d4['AOD_500nm'], d4['IR BCc'])[0]
r_bin = stats.pearsonr(bv.index, bv['BC_mean'])[0] if len(bv) >= 3 else np.nan

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) Daily scatter
ax = axes[0]
for s in SO:
    d = d4[d4['Season'] == s]
    ax.scatter(d['AOD_500nm'], d['IR BCc'], s=12, alpha=0.35, color=SC[s], label=s)
z = np.polyfit(d4['AOD_500nm'], d4['IR BCc'], 1)
xr = np.linspace(d4['AOD_500nm'].min(), d4['AOD_500nm'].max(), 50)
ax.plot(xr, np.poly1d(z)(xr), 'k--', linewidth=1.5, alpha=0.7)
ax.text(0.05, 0.95, f'r = {r_daily:.3f}\nn = {len(d4)}',
       transform=ax.transAxes, va='top', fontsize=10, fontweight='bold',
       bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray', alpha=0.9))
ax.set_xlabel('AOD$_{500}$')
ax.set_ylabel('BC ($\\mu$g/m$^3$)')
ax.set_title('(a) Daily scatter', fontweight='bold')
ax.legend(fontsize=8)

# (b) Binned means
ax = axes[1]
ax.errorbar(bv.index, bv['BC_mean'], yerr=bv['BC_se'], fmt='ko-', markersize=7,
           linewidth=2, capsize=4, zorder=5)
# Annotate count on select bins
for xi, row in bv.iterrows():
    ax.annotate(f'{int(row["n"])}', (xi, row['BC_mean']), textcoords='offset points',
               xytext=(0, 10), fontsize=7, ha='center', color='gray')
z2 = np.polyfit(bv.index, bv['BC_mean'], 1)
ax.plot(bv.index, np.poly1d(z2)(bv.index), 'r--', linewidth=1.5, alpha=0.7)
ax.text(0.05, 0.95, f'r = {r_bin:.3f}\n{len(bv)} bins',
       transform=ax.transAxes, va='top', fontsize=10, fontweight='bold',
       bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray', alpha=0.9))
ax.set_xlabel('AOD$_{500}$ (bin center)')
ax.set_ylabel('Mean BC ($\\mu$g/m$^3$)')
ax.set_title('(b) Binned means ($\\Delta$AOD = 0.05)', fontweight='bold')

# (c) Seasonal binned curves
ax = axes[2]
for s in SO:
    sd = d4[d4['Season'] == s]
    sb = sd.groupby('AOD_mid', observed=True).agg(BC_mean=('IR BCc','mean'), n=('IR BCc','count')).dropna()
    sb = sb[sb['n'] >= 2]
    if len(sb) >= 2:
        ax.plot(sb.index, sb['BC_mean'], 'o-', markersize=6, linewidth=2,
               color=SC[s], label=s)
ax.set_xlabel('AOD$_{500}$ (bin center)')
ax.set_ylabel('Mean BC ($\\mu$g/m$^3$)')
ax.set_title('(c) Seasonal binned BC-AOD', fontweight='bold')
ax.legend(fontsize=9)

plt.suptitle('Figure 4: AOD Binning Analysis (Segura et al.)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT}/fig04_aod_binning.png')
plt.show()

---
## Figure 5: SSA Spectral Shape, dSSA, w vs SSA

In [ ]:
d5 = m.dropna(subset=['SSA_440']).copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) SSA spectral shape by season
ax = axes[0]
wls = [440, 675, 870, 1020]
for s in SO:
    sub = d5[d5['Season'] == s]
    means = [sub[f'SSA_{w}'].mean() for w in wls]
    stds = [sub[f'SSA_{w}'].std() for w in wls]
    n = len(sub)
    if n > 0:
        ax.errorbar(wls, means, yerr=[s_/np.sqrt(n) for s_ in stds],
                   fmt='o-', markersize=8, linewidth=2.5, capsize=5,
                   color=SC[s], label=f'{s} (n={n})')
ax.set_xlabel('Wavelength (nm)', fontsize=10)
ax.set_ylabel('SSA', fontsize=10)
ax.set_title('(a) SSA spectral shape by season', fontweight='bold')
ax.legend(fontsize=8)

# (b) dSSA distribution by season
ax = axes[1]
for s in SO:
    sub = d5[d5['Season'] == s]['dSSA'].dropna()
    ax.hist(sub, bins=20, alpha=0.5, color=SC[s], label=f'{s} (n={len(sub)})', edgecolor='white')
ax.axvline(0, color='k', linewidth=1, linestyle='--', alpha=0.5)
ax.set_xlabel('dSSA = SSA$_{440}$ $-$ SSA$_{870}$', fontsize=10)
ax.set_ylabel('Count', fontsize=10)
ax.set_title('(b) dSSA distribution', fontweight='bold')
ax.legend(fontsize=8)
ax.text(0.05, 0.95, 'dSSA < 0: dust-like\ndSSA > 0: BC-dominated',
       transform=ax.transAxes, va='top', fontsize=8, style='italic',
       bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))

# (c) w vs SSA_440
ax = axes[2]
wssa = dw.set_index('date')[['w','Season']].join(ssa[['SSA_440']], how='inner').dropna()
for s in SO:
    sub = wssa[wssa['Season'] == s]
    ax.scatter(sub['w'], sub['SSA_440'], s=40, alpha=0.6,
              color=SC[s], edgecolors='k', linewidth=0.3, label=s, zorder=3)
if len(wssa) >= 5:
    r, p = stats.pearsonr(wssa['w'], wssa['SSA_440'])
    z = np.polyfit(wssa['w'], wssa['SSA_440'], 1)
    xr = np.linspace(wssa['w'].min(), wssa['w'].max(), 50)
    ax.plot(xr, np.poly1d(z)(xr), 'k--', linewidth=1.5, alpha=0.7, zorder=4)
    ax.text(0.05, 0.05, f'r = {r:.3f}, p = {p:.1e}\nn = {len(wssa)}',
           transform=ax.transAxes, fontsize=9, fontweight='bold',
           bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray', alpha=0.9))
ax.set_xlabel('w = (AS+AN) / (AS+AN+BC)', fontsize=10)
ax.set_ylabel('SSA$_{440}$', fontsize=10)
ax.set_title('(c) w vs SSA$_{440}$ (Eom et al.)', fontweight='bold')
ax.legend(fontsize=8)

plt.suptitle('Figure 5: SSA Analysis (AERONET L1.5 Inversion)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT}/fig05_ssa_analysis.png')
plt.show()

---
## Figure 6: eta Decomposition (Snider et al.)

In [ ]:
d6 = m.dropna(subset=['IR BCc','AOD_500nm']).copy()
d6 = d6[d6['AOD_500nm'] > 0]
d6['eta'] = d6['IR BCc'] / d6['AOD_500nm']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) Seasonal eta bar chart
ax = axes[0]
means, errs, ns = [], [], []
for s in SO:
    sub = d6[d6['Season'] == s]['eta']
    means.append(sub.mean())
    errs.append(sub.std() / np.sqrt(len(sub)))
    ns.append(len(sub))
bars = ax.bar(SO, means, yerr=errs, capsize=5, color=[SC[s] for s in SO],
              edgecolor='black', linewidth=0.5, alpha=0.8, width=0.6)
for i, (n, bar) in enumerate(zip(ns, bars)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + errs[i] + 0.5,
            f'n={n}', ha='center', fontsize=9)
ax.set_ylabel('$\\eta_{BC}$ = BC / AOD$_{500}$ ($\\mu$g m$^{-3}$)', fontsize=10)
ax.set_title('(a) $\\eta$ by season', fontweight='bold')

# (b) eta vs precipitable water
ax = axes[1]
pw = 'Precipitable_Water(cm)'
d6pw = d6.dropna(subset=[pw]) if pw in d6.columns else pd.DataFrame()
if len(d6pw) >= 5:
    for s in SO:
        sub = d6pw[d6pw['Season'] == s]
        ax.scatter(sub[pw], sub['eta'], s=18, alpha=0.45, color=SC[s], label=s, zorder=3)
    r, p = stats.pearsonr(d6pw[pw], d6pw['eta'])
    z = np.polyfit(d6pw[pw], d6pw['eta'], 1)
    xr = np.linspace(d6pw[pw].min(), d6pw[pw].max(), 50)
    ax.plot(xr, np.poly1d(z)(xr), 'k--', linewidth=1.5, alpha=0.7)
    ax.text(0.95, 0.95, f'r = {r:.3f}\nn = {len(d6pw)}',
           transform=ax.transAxes, ha='right', va='top', fontsize=9, fontweight='bold',
           bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray', alpha=0.9))
    ax.set_xlabel('Precipitable Water (cm)', fontsize=10)
    ax.set_ylabel('$\\eta_{BC}$', fontsize=10)
    ax.set_title('(b) $\\eta$ vs precipitable water', fontweight='bold')
    ax.legend(fontsize=8)

# (c) CV comparison
ax = axes[2]
cvs = {
    'BC': d6['IR BCc'].std() / d6['IR BCc'].mean(),
    'AOD$_{500}$': d6['AOD_500nm'].std() / d6['AOD_500nm'].mean(),
    '$\\eta_{BC}$': d6['eta'].std() / d6['eta'].mean(),
}
bars = ax.bar(cvs.keys(), cvs.values(), color=['#E74C3C','#3498DB','#9B59B6'],
              edgecolor='black', linewidth=0.5, alpha=0.8, width=0.5)
for bar, v in zip(bars, cvs.values()):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f'{v:.2f}',
            ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('Coefficient of Variation', fontsize=10)
ax.set_title('(c) Variability decomposition', fontweight='bold')

plt.suptitle('Figure 6: $\\eta$ = PM$_{2.5}$/AOD Decomposition (Snider et al.)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT}/fig06_eta_decomposition.png')
plt.show()

---
## Figure 7: HIPS/FTIR Ratio vs AOD

In [ ]:
d7 = m.dropna(subset=['hips_bc','ftir_ec','AOD_500nm']).copy()
d7['ratio'] = d7['hips_bc'] / d7['ftir_ec']

fig, ax = plt.subplots(figsize=(9, 6.5))

for s in SO:
    sub = d7[d7['Season'] == s]
    ax.scatter(sub['AOD_500nm'], sub['ratio'], s=50, alpha=0.6,
              color=SC[s], edgecolors='k', linewidth=0.3, label=s, zorder=3)

ax.axhline(1, color='red', linewidth=1.5, linestyle='--', alpha=0.6, label='HIPS = FTIR', zorder=2)

if len(d7) >= 5:
    r, p = stats.pearsonr(d7['AOD_500nm'], d7['ratio'])
    z = np.polyfit(d7['AOD_500nm'], d7['ratio'], 1)
    xr = np.linspace(d7['AOD_500nm'].min(), d7['AOD_500nm'].max(), 50)
    ax.plot(xr, np.poly1d(z)(xr), 'k-', linewidth=2, alpha=0.8, zorder=4)
    ax.text(0.05, 0.95, f'r = {r:.3f}, p = {p:.1e}\nn = {len(d7)}\nslope = {z[0]:.2f}',
           transform=ax.transAxes, va='top', fontsize=10, fontweight='bold',
           bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='gray', alpha=0.9))

ax.set_xlabel('AOD$_{500}$', fontsize=12)
ax.set_ylabel('HIPS BC / FTIR EC', fontsize=12)
ax.set_title('Figure 7: HIPS/FTIR Ratio vs AOD$_{500}$ — Addis Ababa',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(f'{OUT}/fig07_hips_ftir_vs_aod.png')
plt.show()

print('HIPS/FTIR ratio by season:')
for s in SO:
    sub = d7[d7['Season'] == s]['ratio']
    print(f'  {s}: {sub.mean():.2f} +/- {sub.std():.2f} (n={len(sub)})')

---
## Figure 8: HIPS vs FTIR Colored by Iron (new)

In [ ]:
# Iron is on different filter IDs than HIPS/FTIR (ChemSpec vs FTIR/HIPS strips),
# but they share dates. Match by date instead of filter ID.
etad = filter_data[filter_data['Site'] == 'ETAD'].copy()
etad['SampleDate'] = pd.to_datetime(etad['SampleDate'])

def get_param_by_date(param):
    sub = etad[etad['Parameter'] == param].dropna(subset=['Concentration'])
    return sub.groupby('SampleDate')['Concentration'].first()

iron_s = get_param_by_date('ChemSpec_Iron_PM2.5')
ftir_s = get_param_by_date('EC_ftir')
hips_s = get_param_by_date('HIPS_Fabs')

d8 = pd.DataFrame({'iron': iron_s, 'ftir_ec': ftir_s, 'hips_fabs': hips_s}).dropna()
d8['hips_bc'] = d8['hips_fabs'] / MAC_VALUE
d8['Season'] = d8.index.month.map(SEASON_MAP)
print(f'Iron + HIPS + FTIR matched by date: {len(d8)} filters')

# Split at median iron
iron_med = d8['iron'].median()
d8['iron_group'] = np.where(d8['iron'] <= iron_med, 'Low Fe', 'High Fe')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# (a) Colored by iron concentration
ax = axes[0]
sc = ax.scatter(d8['ftir_ec'], d8['hips_bc'], c=d8['iron'], cmap='YlOrRd',
               s=50, alpha=0.7, edgecolors='k', linewidth=0.3, zorder=3)
cb = plt.colorbar(sc, ax=ax)
cb.set_label('Iron ($\\mu$g/m$^3$)', fontsize=10)
lim = max(d8['ftir_ec'].max(), d8['hips_bc'].max()) * 1.1
ax.plot([0, lim], [0, lim], 'k--', linewidth=1, alpha=0.4, zorder=2)
if len(d8) >= 5:
    r, p = stats.pearsonr(d8['ftir_ec'], d8['hips_bc'])
    ax.text(0.05, 0.95, f'r = {r:.3f}\nn = {len(d8)}',
           transform=ax.transAxes, va='top', fontsize=9, fontweight='bold',
           bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray', alpha=0.9))
ax.set_xlabel('FTIR EC ($\\mu$g/m$^3$)', fontsize=11)
ax.set_ylabel('HIPS BC ($\\mu$g/m$^3$)', fontsize=11)
ax.set_title('(a) HIPS vs FTIR, colored by filter iron', fontweight='bold')
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)

# (b) Separate regression lines for low vs high iron
ax = axes[1]
colors_fe = {'Low Fe': '#3498DB', 'High Fe': '#E74C3C'}
for grp in ['Low Fe', 'High Fe']:
    sub = d8[d8['iron_group'] == grp]
    ax.scatter(sub['ftir_ec'], sub['hips_bc'], s=40, alpha=0.5,
              color=colors_fe[grp], edgecolors='k', linewidth=0.2, label=grp, zorder=3)
    if len(sub) >= 5:
        slope, intercept, r, p, se = stats.linregress(sub['ftir_ec'], sub['hips_bc'])
        xr = np.linspace(sub['ftir_ec'].min(), sub['ftir_ec'].max(), 50)
        ax.plot(xr, slope * xr + intercept, '-', linewidth=2.5, color=colors_fe[grp],
               alpha=0.8, zorder=4)
        y_pos = 0.95 if grp == 'Low Fe' else 0.78
        ax.text(0.05, y_pos,
               f'{grp}: slope={slope:.2f}, int={intercept:.2f}\nr={r:.3f}, n={len(sub)}',
               transform=ax.transAxes, va='top', fontsize=8, color=colors_fe[grp],
               fontweight='bold',
               bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=colors_fe[grp], alpha=0.8))

ax.plot([0, lim], [0, lim], 'k--', linewidth=1, alpha=0.4, zorder=2, label='1:1')
ax.set_xlabel('FTIR EC ($\\mu$g/m$^3$)', fontsize=11)
ax.set_ylabel('HIPS BC ($\\mu$g/m$^3$)', fontsize=11)
ax.set_title(f'(b) Split at median Fe = {iron_med:.3f} $\\mu$g/m$^3$', fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)

plt.suptitle('Figure 8: HIPS Fabs vs FTIR EC — Iron (Dust) Influence',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT}/fig08_hips_ftir_iron.png')
plt.show()

---
## Figure 9: Control Sites HIPS/FTIR vs AOD

In [ ]:
# Load control site data
control_sites = {
    'Delhi':   {'code': 'INDH', 'aeronet_file': 'data/aeronet_aod_delhi.csv'},
    'Beijing': {'code': 'CHTS', 'aeronet_file': 'data/aeronet_aod_beijing.csv'},
    'JPL':     {'code': 'USPA', 'aeronet_file': 'data/aeronet_aod_jpl.csv'},
}

# Addis regression from Fig 7 (for overlay)
d7_for_line = m.dropna(subset=['hips_bc','ftir_ec','AOD_500nm']).copy()
d7_for_line['ratio'] = d7_for_line['hips_bc'] / d7_for_line['ftir_ec']
addis_z = np.polyfit(d7_for_line['AOD_500nm'], d7_for_line['ratio'], 1) if len(d7_for_line) >= 5 else None

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, (site_name, cfg) in enumerate(control_sites.items()):
    ax = axes[idx]
    site_code = cfg['code']
    aeronet_file = cfg['aeronet_file']
    
    # Get HIPS and FTIR for this site
    site_chem = pivot_filter_by_id(filter_data, site_code,
                                    params=['EC_ftir', 'HIPS_Fabs'])
    if site_chem is None or len(site_chem) == 0:
        ax.text(0.5, 0.5, f'{site_name}\nNo matched\nHIPS+FTIR data',
               transform=ax.transAxes, ha='center', va='center', fontsize=11)
        ax.set_title(f'{site_name}', fontweight='bold')
        continue
    
    site_chem['date'] = pd.to_datetime(site_chem['date'])
    site_chem['hips_bc'] = site_chem['hips_fabs'] / MAC_VALUE
    site_chem = site_chem.dropna(subset=['hips_bc', 'ftir_ec'])
    site_chem['ratio'] = site_chem['hips_bc'] / site_chem['ftir_ec']
    
    # Try to load AERONET for this site
    # Control site files have 5 metadata lines before header (not 6 like Addis)
    has_aod = False
    if os.path.exists(aeronet_file):
        try:
            site_aod = pd.read_csv(aeronet_file, skiprows=5)
            date_col = [c for c in site_aod.columns if 'Date' in c and 'dd' in c]
            if date_col:
                site_aod['Date'] = pd.to_datetime(site_aod[date_col[0]], format='%d:%m:%Y')
                site_aod.set_index('Date', inplace=True)
                site_aod.sort_index(inplace=True)
                site_aod.replace(MISS, np.nan, inplace=True)
                
                # Merge by date
                site_merged = site_chem.set_index('date').join(
                    site_aod[['AOD_500nm']], how='inner').dropna(subset=['AOD_500nm','ratio'])
                has_aod = len(site_merged) >= 5
        except Exception as e:
            print(f'  {site_name} AERONET load error: {e}')
    
    if has_aod:
        ax.scatter(site_merged['AOD_500nm'], site_merged['ratio'], s=35, alpha=0.5,
                  color='#7F8C8D', edgecolors='k', linewidth=0.2, label=site_name, zorder=3)
        r, p = stats.pearsonr(site_merged['AOD_500nm'], site_merged['ratio'])
        # Site regression
        z_site = np.polyfit(site_merged['AOD_500nm'], site_merged['ratio'], 1)
        xr = np.linspace(site_merged['AOD_500nm'].min(), site_merged['AOD_500nm'].max(), 50)
        ax.plot(xr, np.poly1d(z_site)(xr), '-', color='#7F8C8D', linewidth=2, alpha=0.7)
        ax.text(0.05, 0.95, f'{site_name}: r = {r:.3f}\np = {p:.2e}\nn = {len(site_merged)}',
               transform=ax.transAxes, va='top', fontsize=9, fontweight='bold',
               bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray', alpha=0.9))
        x_axis_max = max(site_merged['AOD_500nm'].max(), 0.5)
    else:
        # No AOD: show HIPS/FTIR ratio distribution instead
        ax.hist(site_chem['ratio'].dropna(), bins=20, color='#7F8C8D', alpha=0.6,
                edgecolor='white')
        med = site_chem['ratio'].median()
        ax.axvline(med, color='k', linewidth=1.5, linestyle='-')
        ax.text(0.05, 0.95, f'n = {len(site_chem)}\nmedian = {med:.2f}\n(no AOD data)',
               transform=ax.transAxes, va='top', fontsize=9, fontweight='bold',
               bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray', alpha=0.9))
        ax.set_xlabel('HIPS BC / FTIR EC')
        ax.set_ylabel('Count')
        ax.set_title(f'{site_name}', fontweight='bold')
        continue
    
    # Overlay Addis regression line
    if addis_z is not None:
        xr_addis = np.linspace(0, x_axis_max, 50)
        ax.plot(xr_addis, np.poly1d(addis_z)(xr_addis), '--', color='#F39C12',
               linewidth=2, alpha=0.8, label='Addis regression', zorder=5)
    
    ax.axhline(1, color='red', linewidth=1, linestyle=':', alpha=0.5)
    ax.set_xlabel('AOD$_{500}$', fontsize=10)
    ax.set_ylabel('HIPS BC / FTIR EC', fontsize=10)
    ax.set_title(f'{site_name}', fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Figure 9: Control Sites — HIPS/FTIR Ratio vs AOD (with Addis regression overlay)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT}/fig09_control_sites.png')
plt.show()

---
## Figure 10: Season-Separated HIPS vs FTIR Regressions

In [ ]:
d10 = m.dropna(subset=['hips_bc','ftir_ec']).copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True, sharex=True)

lim = max(d10['ftir_ec'].max(), d10['hips_bc'].max()) * 1.1

for idx, s in enumerate(SO):
    ax = axes[idx]
    sub = d10[d10['Season'] == s].copy()
    
    ax.scatter(sub['ftir_ec'], sub['hips_bc'], s=40, alpha=0.5,
              color=SC[s], edgecolors='k', linewidth=0.3, zorder=3)
    ax.plot([0, lim], [0, lim], 'k--', linewidth=1, alpha=0.4, zorder=2)
    
    if len(sub) >= 5:
        slope, intercept, r, p, se = stats.linregress(sub['ftir_ec'], sub['hips_bc'])
        xr = np.linspace(0, sub['ftir_ec'].max() * 1.05, 50)
        ax.plot(xr, slope * xr + intercept, '-', linewidth=2.5, color=SC[s],
               alpha=0.9, zorder=4)
        
        ax.text(0.05, 0.95,
               f'slope = {slope:.2f}\nintercept = {intercept:.2f}\nr = {r:.3f}\nn = {len(sub)}',
               transform=ax.transAxes, va='top', fontsize=9, fontweight='bold',
               bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=SC[s], alpha=0.9, linewidth=1.5))
    
    ax.set_xlabel('FTIR EC ($\\mu$g/m$^3$)', fontsize=11)
    if idx == 0:
        ax.set_ylabel('HIPS BC ($\\mu$g/m$^3$)', fontsize=11)
    ax.set_title(f'{s}', fontweight='bold', fontsize=12, color=SC[s])
    ax.set_xlim(0, lim)
    ax.set_ylim(0, lim)
    ax.set_aspect('equal')

plt.suptitle('Figure 10: Season-Separated HIPS vs FTIR Regressions — Addis Ababa',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT}/fig10_seasonal_hips_ftir.png')
plt.show()

# Summary table
print('Season-separated regression summary:')
print(f'{"Season":<10} {"slope":>8} {"intercept":>10} {"r":>8} {"n":>5}')
for s in SO:
    sub = d10[d10['Season'] == s]
    if len(sub) >= 5:
        slope, intercept, r, p, se = stats.linregress(sub['ftir_ec'], sub['hips_bc'])
        print(f'{s:<10} {slope:>8.2f} {intercept:>10.2f} {r:>8.3f} {len(sub):>5}')

---
## Summary

All 10 figures saved to `output/plots/pub_figures/`.